# Events vs. Note table: does the events table contain notes? + how the join works

Standalone demo (does not touch `testing_events_parse.ipynb`). Parses one Bach chorale MEI,
then answers two things empirically:
1. Is the events table notes-only, non-notes-only, or both?
2. How do you join an event (a lyric) back to the note it sits on?

In [15]:
import pandas as pd
from camat.parser_registry import parse_files

FILE_SOURCES = [
    'https://raw.githubusercontent.com/music-encoding/sample-encodings/main/MEI_5.0/Music/Complete_examples/Bach-JS_Ein_feste_Burg.mei',
]

# Parse only — no plotting, no previews, keep it quiet.
results, dfs_by_name, df_processed = parse_files(
    FILE_SOURCES,
    parsing_backend='partitura',
    backend='none',
    display_preview_df_pitch=False,
    display_preview_df_events=False,
    print_parsed_summary=False,
    parse_enharmonic=True,
    include_xml_ids=True,
    quiet_native_warnings=True,
    show_progress=False,
)

names = list(dfs_by_name)
pitch_name = next(n for n in names if n.endswith('_pitch'))
events_name = next(n for n in names if n.endswith('_events'))
df_pitch = dfs_by_name[pitch_name]
df_events = dfs_by_name[events_name]
print('tables:', names)
print('df_pitch  shape:', df_pitch.shape)
print('df_events shape:', df_events.shape)

Processing (partitura): Bach-JS_Ein_feste_Burg.mei -> 00_bach_js_ein_feste_burg
Detected common-notation MEI. Skipping mensural duration/meter preprocessing.
Dropped weaker duplicate MEI text events: 10 row(s) without usable anchors.
Extracted non-barline MEI events: 99 event(s), types=['fermata', 'lyric', 'measure', 'slur', 'tie'].
tables: ['00_bach_js_ein_feste_burg_pitch', '00_bach_js_ein_feste_burg_events']
df_pitch  shape: (235, 17)
df_events shape: (99, 34)


## Q: Does the events table contain notes?
Look at the distinct `type` values in the events table, and check explicitly for any `note` rows.

In [16]:
print('Distinct event types in df_events:')
print(df_events['type'].value_counts(dropna=False))

note_like = df_events['type'].astype(str).str.lower().isin(['note', 'chord'])
print('\nRows in df_events whose type is note/chord:', int(note_like.sum()))
print('Does df_events have a Pitch/MIDI column?',
      'Pitch' in df_events.columns, '/', 'MIDI' in df_events.columns)
print('Does df_pitch have a Pitch/MIDI column?',
      'Pitch' in df_pitch.columns, '/', 'MIDI' in df_pitch.columns)

Distinct event types in df_events:
type
lyric      58
measure    16
fermata    13
slur       11
tie         1
Name: count, dtype: int64

Rows in df_events whose type is note/chord: 0
Does df_events have a Pitch/MIDI column? False / False
Does df_pitch have a Pitch/MIDI column? True / True


## The join: lyric event → the note it sits on
Events link to notes via `start_xml_id` (event) ↔ `xml_id` (note). We are NOT joining on row number.

In [17]:
lyrics = df_events[df_events['type'] == 'lyric'].copy()

joined = lyrics.merge(
    df_pitch[['xml_id', 'Measure', 'Global Onset', 'Pitch', 'MIDI', 'Voice', 'Duration']],
    left_on='start_xml_id', right_on='xml_id',
    how='left', suffixes=('_event', '_note'),
)

cols = ['text', 'verse_n', 'start_xml_id',
        'Global Onset_event', 'Global Onset_note',
        'Pitch', 'MIDI', 'Voice_note']
out = joined[cols].rename(columns={
    'Global Onset_event': 'event_onset',
    'Global Onset_note': 'note_onset',
    'Voice_note': 'note_voice',
})

matched = out['Pitch'].notna().sum()
print(f'Lyric rows: {len(out)} | matched to a note via xml_id: {matched}')
with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 200):
    display(out.head(30))

Lyric rows: 58 | matched to a note via xml_id: 58


,text,verse_n,start_xml_id,event_onset,note_onset,Pitch,MIDI,note_voice
0,Ein´,1,d1e64,-1.0,-1.0,D5,74,P1 - Voice 1
1,Er,2,d1e64,-1.0,-1.0,D5,74,P1 - Voice 1
2,hilft,2,d1e366,0.0,0.0,D5,74,P1 - Voice 1
3,fe,1,d1e366,0.0,0.0,D5,74,P1 - Voice 1
4,uns,2,d1e390,1.0,1.0,D5,74,P1 - Voice 1
5,ste,1,d1e390,1.0,1.0,D5,74,P1 - Voice 1
6,frei,2,d1e414,2.0,2.0,A4,69,P1 - Voice 1
7,Burg,1,d1e414,2.0,2.0,A4,69,P1 - Voice 1
8,aus,2,d1e458,3.0,3.0,C5,72,P1 - Voice 1
9,ist,1,d1e458,3.0,3.0,C5,72,P1 - Voice 1


## Proof that row index is NOT the alignment
Take a real lyric event at position `i` in the events table. The note at the *same* row `i` of the
note table is a different, unrelated note — the real partner is found only by matching
`start_xml_id` → `xml_id`. (For contrast, a `measure` event anchors to no single note at all,
so matching its empty `start_xml_id` correctly returns nothing.)

In [18]:
# Pick the first lyric event that actually anchors to a note (has a start_xml_id)
lyric_anchored = (df_events['type'] == 'lyric') & df_events['start_xml_id'].notna()
i = int(lyric_anchored.idxmax())
ev = df_events.iloc[i]
print(f'Events row {i}: type={ev["type"]}, text={ev["text"]!r}, '
      f'start_xml_id={ev["start_xml_id"]}, Global Onset={ev["Global Onset"]}')

# Same row index in the NOTE table -> an unrelated note
same_row_note = df_pitch.iloc[i]
print(f'\nPitch row {i} (same index): xml_id={same_row_note["xml_id"]}, '
      f'Pitch={same_row_note["Pitch"]}, Global Onset={same_row_note["Global Onset"]}  '
      f'<- NOT the lyric\'s note')

# The real partner, found by matching start_xml_id -> xml_id
partner = df_pitch[df_pitch['xml_id'] == ev['start_xml_id']]
print(f'\nActual partner note (matched by xml_id == {ev["start_xml_id"]}):')
display(partner[['xml_id', 'Measure', 'Global Onset', 'Pitch', 'MIDI', 'Voice']])

# Contrast: a measure event has no single-note anchor at all
m = df_events[df_events['type'] == 'measure'].iloc[0]
print(f'Contrast — first measure event: start_xml_id={m["start_xml_id"]} '
      f'(Global Onset={m["Global Onset"]}) -> anchors to no note.')

Events row 1: type=lyric, text='Ein´', start_xml_id=d1e64, Global Onset=-1.0

Pitch row 1 (same index): xml_id=d1e92, Pitch=F4, Global Onset=-1.0  <- NOT the lyric's note

Actual partner note (matched by xml_id == d1e64):


,xml_id,Measure,Global Onset,Pitch,MIDI,Voice
3,d1e64,1,-1.0,D5,74,P1 - Voice 1


Contrast — first measure event: start_xml_id=<NA> (Global Onset=-1.0) -> anchors to no note.


## Bonus: stable, deterministic verse ordering
The events table is sorted **only** by `Global Onset` using pandas' default (unstable) sort, so when
verse 1 and verse 2 share the same note (same onset) their order is arbitrary tie-breaking — that's the
wandering `1,2 / 2,1 / 2,1 / 1,2 ...` you noticed. Re-sorting with an explicit secondary key and
`kind="stable"` makes it deterministic. (`verse_n` is stored as a string, so for ≥10 verses sort on
`verse_n.astype(int)` instead.)

In [19]:
# As-returned, the events table is sorted ONLY by Global Onset, with pandas' default
# (unstable) sort and no secondary key -> verse order on a shared note is arbitrary.
# Re-sort explicitly with a secondary key and kind='stable' to make it deterministic.
asret = df_events[df_events['type'] == 'lyric'].copy()

lyrics_stable = asret.sort_values(
    ['Global Onset', 'verse_n'], kind='stable'
).reset_index(drop=True)

print('Within-onset verse order, as-returned vs. stable [Global Onset, verse_n]:')
for onset in sorted(asret['Global Onset'].dropna().unique())[:8]:
    a = list(asret.loc[asret['Global Onset'] == onset, 'verse_n'])
    b = list(lyrics_stable.loc[lyrics_stable['Global Onset'] == onset, 'verse_n'])
    print(f'  onset {onset:>5}: as-returned {a}  ->  stable {b}')

print('\nStably sorted lyrics (first 12 rows):')
with pd.option_context('display.max_rows', 12, 'display.max_columns', None, 'display.width', 200):
    display(lyrics_stable[['Global Onset', 'verse_n', 'text', 'start_xml_id']].head(12))

Within-onset verse order, as-returned vs. stable [Global Onset, verse_n]:
  onset  -1.0: as-returned ['1', '2']  ->  stable ['1', '2']
  onset   0.0: as-returned ['2', '1']  ->  stable ['1', '2']
  onset   1.0: as-returned ['2', '1']  ->  stable ['1', '2']
  onset   2.0: as-returned ['2', '1']  ->  stable ['1', '2']
  onset   3.0: as-returned ['2', '1']  ->  stable ['1', '2']
  onset   4.0: as-returned ['2', '1']  ->  stable ['1', '2']
  onset   5.0: as-returned ['1', '2']  ->  stable ['1', '2']
  onset   6.0: as-returned ['1', '2']  ->  stable ['1', '2']

Stably sorted lyrics (first 12 rows):


,Global Onset,verse_n,text,start_xml_id
0,-1.0,1,Ein´,d1e64
1,-1.0,2,Er,d1e64
2,0.0,1,fe,d1e366
3,0.0,2,hilft,d1e366
4,1.0,1,ste,d1e390
5,1.0,2,uns,d1e390
6,2.0,1,Burg,d1e414
7,2.0,2,frei,d1e414
8,3.0,1,ist,d1e458
9,3.0,2,aus,d1e458
